In [60]:
content_recommend("Interstellar", 5)

,title,similarity
0,Space Station 3D,17.8
1,Mission to Mir,14.3
2,Contact,13.3
3,Star Wreck: In the Pirkinning,13.2
4,The Martian,13.1


In [59]:
content_recommend("interstellar", 5)

,title,similarity
0,Space Station 3D,17.8
1,Mission to Mir,14.3
2,Contact,13.3
3,Star Wreck: In the Pirkinning,13.2
4,The Martian,13.1


In [58]:
def content_recommend(movie_title, n=10):
    
    matches = content_data[
        content_data["title_clean"] == movie_title.strip().lower()
    ]

    if matches.empty:
        return pd.DataFrame()

    movie_index = matches.index[0]

    distances, indices = content_model.kneighbors(
        tfidf_matrix[movie_index],
        n_neighbors=n + 1
    )

    recommendations = []

    for distance, index in zip(
        distances[0][1:],
        indices[0][1:]
    ):
        
        similarity = 1 - distance

        recommendations.append({
            "title": content_data.iloc[index]["title"],
            "similarity": round(similarity * 100, 1)
        })

    return pd.DataFrame(recommendations)

In [57]:
content_data["title_clean"] = (
    content_data["title"]
    .fillna("")
    .str.strip()
    .str.lower()
)

print("Title lookup column created!")

Title lookup column created!


In [56]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    tfidf,
    "../models/content_tfidf.pkl"
)

joblib.dump(
    content_model,
    "../models/content_knn.pkl"
)

content_data.to_pickle(
    "../models/content_data.pkl"
)

print("Content recommendation artifacts saved!")

Content recommendation artifacts saved!


In [55]:
content_recommend("Interstellar", 10)

,title,similarity
0,Space Station 3D,17.8
1,Mission to Mir,14.3
2,Contact,13.3
3,Star Wreck: In the Pirkinning,13.2
4,The Martian,13.1
5,First Spaceship on Venus,13.1
6,Manhunt in Space,13.0
7,Hubble 3D,12.9
8,Journey to the Seventh Planet,12.8
9,Destiny in Space,12.7


In [54]:
def content_recommend(movie_title, n=10):
    
    matches = content_data[
        content_data["title"].str.lower() == movie_title.lower()
    ]

    if matches.empty:
        return pd.DataFrame()

    movie_index = matches.index[0]

    distances, indices = content_model.kneighbors(
        tfidf_matrix[movie_index],
        n_neighbors=n + 1
    )

    recommendations = []

    for distance, index in zip(distances[0][1:], indices[0][1:]):
        
        similarity = 1 - distance

        recommendations.append({
            "title": content_data.iloc[index]["title"],
            "similarity": round(similarity * 100, 1)
        })

    return pd.DataFrame(recommendations)

In [53]:
content_recommend("Interstellar", 10)

,title,similarity
0,Space Station 3D,0.178
1,Mission to Mir,0.143
2,Contact,0.133
3,Star Wreck: In the Pirkinning,0.132
4,The Martian,0.131
5,First Spaceship on Venus,0.131
6,Manhunt in Space,0.130
7,Hubble 3D,0.129
8,Journey to the Seventh Planet,0.128
9,Destiny in Space,0.127


In [52]:
content_recommend("Toy Story", 10)

,title,similarity
0,Toy Story 2,0.467
1,Toy Story 3,0.452
2,Small Fry,0.245
3,The 40 Year Old Virgin,0.214
4,Toy Story of Terror!,0.200
5,Hot Splash,0.196
6,Andy Hardy's Blonde Trouble,0.196
7,Partysaurus Rex,0.196
8,Andy Kaufman Plays Carnegie Hall,0.189
9,Toy Story That Time Forgot,0.176


In [51]:

def content_recommend(movie_title, n=10):
    
    matches = content_data[
        content_data["title"].str.lower() == movie_title.lower()
    ]

    if matches.empty:
        return f"Movie '{movie_title}' not found."

    movie_index = matches.index[0]

    distances, indices = content_model.kneighbors(
        tfidf_matrix[movie_index],
        n_neighbors=n + 1
    )

    recommendations = []

    for distance, index in zip(distances[0][1:], indices[0][1:]):
        
        recommendations.append({
            "title": content_data.iloc[index]["title"],
            "similarity": round(1 - distance, 3)
        })

    return pd.DataFrame(recommendations)

In [50]:
from sklearn.neighbors import NearestNeighbors

content_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=21,
    n_jobs=-1
)

content_model.fit(tfidf_matrix)

print("Content-based recommendation model trained!")

Content-based recommendation model trained!


In [49]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf.fit_transform(
    content_data["content_features"]
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (45408, 50000)


In [48]:
# Prepare content data

content_data = content_data[
    ["id", "title", "content_features"]
].copy()

# Remove duplicate movies
content_data = content_data.drop_duplicates(subset=["id"])

# Remove movies without usable content
content_data = content_data[
    content_data["content_features"].str.strip() != ""
]

content_data = content_data.reset_index(drop=True)

print("Movies available:", len(content_data))
print("Missing content:", content_data["content_features"].isna().sum())

content_data.head()

Movies available: 45408
Missing content: 0


,id,title,content_features
0,862,Toy Story,animation comedy family jealousy toy boy frien...
1,8844,Jumanji,adventure fantasy family board_game disappeara...
2,15602,Grumpier Old Men,romance comedy fishing best_friend duringcredi...
3,31357,Waiting to Exhale,comedy drama romance based_on_novel interracia...
4,11862,Father of the Bride Part II,comedy baby midlife_crisis confidence aging da...


In [47]:
content_data[
    ["id", "title", "content_features"]
].head(5)

,id,title,content_features
0,862,Toy Story,animation comedy family jealousy toy boy frien...
1,8844,Jumanji,adventure fantasy family board_game disappeara...
2,15602,Grumpier Old Men,romance comedy fishing best_friend duringcredi...
3,31357,Waiting to Exhale,comedy drama romance based_on_novel interracia...
4,11862,Father of the Bride Part II,comedy baby midlife_crisis confidence aging da...


In [46]:
content_data["content_features"] = (
    content_data["genres"].apply(lambda x: " ".join(x))
    + " "
    + content_data["keywords"].apply(lambda x: " ".join(x))
    + " "
    + content_data["overview"]
    + " "
    + content_data["cast"].apply(lambda x: " ".join(x))
    + " "
    + content_data["director"]
)

In [45]:
content_data["overview"] = content_data["overview"].fillna("")
content_data["keywords"] = content_data["keywords"].apply(
    lambda x: x if isinstance(x, list) else []
)
content_data["cast"] = content_data["cast"].apply(
    lambda x: x if isinstance(x, list) else []
)
content_data["genres"] = content_data["genres"].apply(
    lambda x: x if isinstance(x, list) else []
)
content_data["director"] = content_data["director"].fillna("")

In [44]:
print("Rows:", len(content_data))
print("Columns:", content_data.columns.tolist())

Rows: 46629
Columns: ['id', 'title', 'genres', 'overview', 'keywords', 'cast', 'director']


In [43]:
content_data = movie_features.merge(
    keywords_features,
    on="id",
    how="left"
)

content_data = content_data.merge(
    credits_features,
    on="id",
    how="left"
)

In [42]:
credits_features = credits.copy()

credits_features["cast"] = credits_features["cast"].apply(
    extract_cast
)

credits_features["director"] = credits_features["crew"].apply(
    extract_director
)

credits_features = credits_features[
    ["id", "cast", "director"]
]

In [41]:
keywords_features = keywords.copy()

keywords_features["keywords"] = keywords_features["keywords"].apply(
    extract_names
)

keywords_features = keywords_features[
    ["id", "keywords"]
]

In [40]:
movie_features["genres"] = movie_features["genres"].apply(extract_names)

In [39]:
movie_features = movies_clean[
    ["id", "title", "genres", "overview"]
].copy()

movie_features.head()

,id,title,genres,overview
0,862,Toy Story,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","Led by Woody, Andy's toys live happily in his ..."
1,8844,Jumanji,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",When siblings Judy and Peter discover an encha...
2,15602,Grumpier Old Men,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",A family wedding reignites the ancient feud be...
3,31357,Waiting to Exhale,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...","Cheated on, mistreated and stepped on, the wom..."
4,11862,Father of the Bride Part II,"[{'id': 35, 'name': 'Comedy'}]",Just when George Banks has recovered from his ...


In [38]:
print(extract_cast(credits.iloc[0]["cast"]))
print(extract_director(credits.iloc[0]["crew"]))

['tom_hanks', 'tim_allen', 'don_rickles', 'jim_varney', 'wallace_shawn']
john_lasseter


In [37]:
def extract_director(value):
    items = parse_json_column(value)

    for item in items:
        if (
            isinstance(item, dict)
            and item.get("job") == "Director"
            and "name" in item
        ):
            return item["name"].replace(" ", "_").lower()

    return ""

In [36]:
def extract_cast(value, top_n=5):
    items = parse_json_column(value)

    return [
        item["name"].replace(" ", "_").lower()
        for item in items[:top_n]
        if isinstance(item, dict) and "name" in item
    ]

In [35]:
print(extract_names(movies_clean.iloc[0]["genres"]))
print(extract_names(keywords.iloc[0]["keywords"]))

['animation', 'comedy', 'family']
['jealousy', 'toy', 'boy', 'friendship', 'friends', 'rivalry', 'boy_next_door', 'new_toy', 'toy_comes_to_life']


In [34]:
def extract_names(value):
    items = parse_json_column(value)

    return [
        item["name"].replace(" ", "_").lower()
        for item in items
        if isinstance(item, dict) and "name" in item
    ]

In [33]:
def extract_names(value):
    items = parse_json_column(value)

    return [
        item["name"].replace(" ", "").lower()
        for item in items
        if isinstance(item, dict) and "name" in item
    ]

In [32]:
import ast

def parse_json_column(value):
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return []

In [31]:
print(credits.iloc[0]["cast"][:500])

[{'cast_id': 14, 'character': 'Woody (voice)', 'credit_id': '52fe4284c3a36847f8024f95', 'gender': 2, 'id': 31, 'name': 'Tom Hanks', 'order': 0, 'profile_path': '/pQFoyx7rp09CJTAb932F2g8Nlho.jpg'}, {'cast_id': 15, 'character': 'Buzz Lightyear (voice)', 'credit_id': '52fe4284c3a36847f8024f99', 'gender': 2, 'id': 12898, 'name': 'Tim Allen', 'order': 1, 'profile_path': '/uX2xVf6pMmPepxnvFWyBtjexzgY.jpg'}, {'cast_id': 16, 'character': 'Mr. Potato Head (voice)', 'credit_id': '52fe4284c3a36847f8024f9d'


In [30]:
print(keywords.iloc[0]["keywords"])

[{'id': 931, 'name': 'jealousy'}, {'id': 4290, 'name': 'toy'}, {'id': 5202, 'name': 'boy'}, {'id': 6054, 'name': 'friendship'}, {'id': 9713, 'name': 'friends'}, {'id': 9823, 'name': 'rivalry'}, {'id': 165503, 'name': 'boy next door'}, {'id': 170722, 'name': 'new toy'}, {'id': 187065, 'name': 'toy comes to life'}]


In [29]:
print("Keywords columns:")
print(keywords.columns.tolist())

print("\nCredits columns:")
print(credits.columns.tolist())

Keywords columns:
['id', 'keywords']

Credits columns:
['cast', 'crew', 'id']


In [28]:
credits.head()

,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862


In [27]:
keywords.head()

,id,keywords
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,8844,"[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,15602,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."
3,31357,"[{'id': 818, 'name': 'based on novel'}, {'id':..."
4,11862,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n..."


In [26]:
keywords = pd.read_csv("../data/raw/keywords.csv")
credits = pd.read_csv("../data/raw/credits.csv")

print("Keywords:", keywords.shape)
print("Credits:", credits.shape)

Keywords: (46419, 2)
Credits: (45476, 3)


In [25]:
movie_ratings = ratings_mapped.merge(
    movies_clean,
    left_on="tmdbId",
    right_on="id",
    how="inner"
)

print("Final joined rows:", len(movie_ratings))
print("Unique movies:", movie_ratings["tmdbId"].nunique())
print("Unique users:", movie_ratings["userId"].nunique())

Final joined rows: 99850
Unique movies: 9025
Unique users: 671


In [24]:
ratings_mapped = ratings.merge(
    links_clean[["movieId", "tmdbId"]],
    on="movieId",
    how="inner"
)

print("Ratings after ID mapping:", len(ratings_mapped))
print("Unique movies:", ratings_mapped["tmdbId"].nunique())

Ratings after ID mapping: 99898
Unique movies: 9048


In [23]:
links_clean = links.dropna(subset=["tmdbId"]).copy()
movies_clean = movies.dropna(subset=["id"]).copy()

links_clean["tmdbId"] = links_clean["tmdbId"].astype(int)
movies_clean["id"] = movies_clean["id"].astype(int)

In [22]:
links["tmdbId"] = pd.to_numeric(links["tmdbId"], errors="coerce")

movies["id"] = pd.to_numeric(movies["id"], errors="coerce")

In [21]:
links[["movieId", "tmdbId"]].head(10)

,movieId,tmdbId
0,1,862.0
1,2,8844.0
2,3,15602.0
3,4,31357.0
4,5,11862.0
5,6,949.0
6,7,11860.0
7,8,45325.0
8,9,9091.0
9,10,710.0


In [20]:
print("Unique MovieLens IDs:", links["movieId"].nunique())
print("Unique TMDB IDs:", links["tmdbId"].nunique())
print("Missing TMDB IDs:", links["tmdbId"].isna().sum())

Unique MovieLens IDs: 45843
Unique TMDB IDs: 45594
Missing TMDB IDs: 219


In [19]:
print("Links rows:", links.shape[0])
print("Links columns:", links.shape[1])
print(links.columns.tolist())

Links rows: 45843
Links columns: 3
['movieId', 'imdbId', 'tmdbId']


In [18]:
links = pd.read_csv("../data/raw/links.csv")

links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [17]:
metadata_ids = set(movies["id"].astype(str))

rating_ids = set(ratings["movieId"].astype(str))

overlap = metadata_ids.intersection(rating_ids)

print("Movies in metadata:", len(metadata_ids))
print("Movies in ratings:", len(rating_ids))
print("Common movie IDs:", len(overlap))

Movies in metadata: 45436
Movies in ratings: 9066
Common movie IDs: 2830


In [16]:
print("Example metadata IDs:")
print(movies["id"].head(10).tolist())

print("\nExample rating movie IDs:")
print(ratings["movieId"].head(10).tolist())

Example metadata IDs:
['862', '8844', '15602', '31357', '11862', '949', '11860', '45325', '9091', '710']

Example rating movie IDs:
[31, 1029, 1061, 1129, 1172, 1263, 1287, 1293, 1339, 1343]


In [15]:
print("Movie metadata ID type:", movies["id"].dtype)
print("Ratings movieId type:", ratings["movieId"].dtype)

Movie metadata ID type: str
Ratings movieId type: int64


In [14]:
print("Unique users:", ratings["userId"].nunique())
print("Unique movies rated:", ratings["movieId"].nunique())
print("Average rating:", ratings["rating"].mean())

Unique users: 671
Unique movies rated: 9066
Average rating: 3.543608255669773


In [13]:
print(ratings.columns.tolist())

['userId', 'movieId', 'rating', 'timestamp']


In [12]:
print("Rating rows:", ratings.shape[0])
print("Rating columns:", ratings.shape[1])

Rating rows: 100004
Rating columns: 4


In [11]:
ratings = pd.read_csv("../data/raw/ratings_small.csv")

ratings.head()

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


In [10]:
movies[[
    "id",
    "title",
    "genres",
    "overview"
]].head(10)

,id,title,genres,overview
0,862,Toy Story,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","Led by Woody, Andy's toys live happily in his ..."
1,8844,Jumanji,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",When siblings Judy and Peter discover an encha...
2,15602,Grumpier Old Men,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",A family wedding reignites the ancient feud be...
3,31357,Waiting to Exhale,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...","Cheated on, mistreated and stepped on, the wom..."
4,11862,Father of the Bride Part II,"[{'id': 35, 'name': 'Comedy'}]",Just when George Banks has recovered from his ...
5,949,Heat,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...","Obsessive master thief, Neil McCauley leads a ..."
6,11860,Sabrina,"[{'id': 35, 'name': 'Comedy'}, {'id': 10749, '...",An ugly duckling having undergone a remarkable...
7,45325,Tom and Huck,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","A mischievous young boy, Tom Sawyer, witnesses..."
8,9091,Sudden Death,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",International action superstar Jean Claude Van...
9,710,GoldenEye,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",James Bond must unmask the mysterious head of ...


In [9]:
print(movies.columns.tolist())

['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']


In [8]:
print("Rows:", movies.shape[0])
print("Columns:", movies.shape[1])

Rows: 45466
Columns: 24


In [7]:
print("Duplicate rows:", movies.duplicated().sum())

Duplicate rows: 13


In [6]:
missing = movies.isnull().sum()

missing[missing > 0].sort_values(ascending=False)

belongs_to_collection    40972
homepage                 37684
tagline                  25054
overview                   954
poster_path                386
runtime                    263
release_date                87
status                      87
imdb_id                     17
original_language           11
revenue                      6
title                        6
video                        6
vote_average                 6
spoken_languages             6
vote_count                   6
popularity                   5
production_companies         3
production_countries         3
dtype: int64

In [5]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  str    
 1   belongs_to_collection  4494 non-null   str    
 2   budget                 45466 non-null  str    
 3   genres                 45466 non-null  str    
 4   homepage               7782 non-null   str    
 5   id                     45466 non-null  str    
 6   imdb_id                45449 non-null  str    
 7   original_language      45455 non-null  str    
 8   original_title         45466 non-null  str    
 9   overview               44512 non-null  str    
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  str    
 12  production_companies   45463 non-null  str    
 13  production_countries   45463 non-null  str    
 14  release_date           45379 non-null  str    
 15  revenue      

In [4]:
print(movies.columns.tolist())

['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']


In [3]:
print("Rows:", movies.shape[0])
print("Columns:", movies.shape[1])

Rows: 45466
Columns: 24


In [2]:
movies = pd.read_csv("../data/raw/movies_metadata.csv")

movies.head()

C:\Users\shrey\AppData\Local\Temp\ipykernel_18764\379100368.py:1: DtypeWarning: Columns (10: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  movies = pd.read_csv("../data/raw/movies_metadata.csv")


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [1]:
import pandas as pd
import numpy as np

print("Environment ready!")

Environment ready!
